# Lasso Regression (L1 Regularization) - Starter Notebook
Lasso adds an L1 penalty that shrinks some coefficients exactly to zero, performing automatic feature selection.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, LassoCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.datasets import make_regression

In [ ]:
# Dataset with many features, only 5 are truly informative
np.random.seed(42)
X, y, true_coef = make_regression(n_samples=200, n_features=20, n_informative=5,
                                   noise=10, coef=True, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print('True informative coefficients (non-zero):', np.sum(true_coef != 0))
print('Feature shape:', X_train.shape)

In [ ]:
# Fit OLS, Ridge, and Lasso
alphas = [0.01, 0.1, 1.0, 10.0, 50.0]

ols = LinearRegression().fit(X_train_s, y_train)
print(f'OLS  non-zero coefs: {np.sum(ols.coef_ != 0):2d}  test MSE: {mean_squared_error(y_test, ols.predict(X_test_s)):.2f}')

for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=5000).fit(X_train_s, y_train)
    nz = np.sum(lasso.coef_ != 0)
    mse = mean_squared_error(y_test, lasso.predict(X_test_s))
    print(f'Lasso α={alpha:<6} non-zero coefs: {nz:2d}  test MSE: {mse:.2f}')

In [ ]:
# Coefficient shrinkage path
alphas_path = np.logspace(-2, 2, 100)
coefs = [Lasso(alpha=a, max_iter=5000).fit(X_train_s, y_train).coef_ for a in alphas_path]

plt.figure(figsize=(8, 4))
plt.semilogx(alphas_path, coefs)
plt.xlabel('Alpha (log scale)')
plt.ylabel('Coefficient value')
plt.title('Lasso Coefficient Path (L1 Regularization)')
plt.axvline(1.0, color='red', linestyle='--', label='α=1')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# CV-selected alpha via LassoCV
lasso_cv = LassoCV(cv=5, max_iter=10000, random_state=42).fit(X_train_s, y_train)
print(f'\nLassoCV best alpha: {lasso_cv.alpha_:.4f}')
print(f'LassoCV non-zero coefs: {np.sum(lasso_cv.coef_ != 0)}')
print(f'LassoCV test MSE: {mean_squared_error(y_test, lasso_cv.predict(X_test_s)):.2f}')

# Compare true vs estimated coefficients
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.stem(true_coef, linefmt='b-', markerfmt='bo', basefmt='k-')
ax1.set_title('True Coefficients')
ax1.set_xlabel('Feature index')

ax2.stem(lasso_cv.coef_, linefmt='r-', markerfmt='ro', basefmt='k-')
ax2.set_title(f'Lasso Estimated Coefs (α={lasso_cv.alpha_:.4f})')
ax2.set_xlabel('Feature index')

plt.tight_layout()
plt.show()